In [1]:
print("ok")

ok


In [21]:
from dotenv import load_dotenv
from pydantic import BaseModel

In [2]:
from langchain_groq import ChatGroq

In [9]:
llm=ChatGroq(model="openai/gpt-oss-120b")

In [10]:
llm.invoke("hi")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond politely. No special instructions. Just greet.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 72, 'total_tokens': 110, 'completion_time': 0.078875246, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.01404, 'prompt_tokens_details': None, 'queue_time': 0.342513855, 'total_time': 0.092915246}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_77b12279f9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fe2e-7244-75a3-bb8c-3f8f6e68fe5d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 38, 'total_tokens': 110, 'output_token_details': {'reasoning': 20}})

In [11]:
from langchain.tools import tool

In [25]:
# recommended way
@tool
def multiply(a: int, b: int) -> int:
    """
    Multiplies two numbers.

    Args:
        a (int): The first number.
        b (int): The second number.
    
    Returns:
        int: The product of the two numbers.
    """
    return a * b

In [13]:
multiply

StructuredTool(name='multiply', description='Multiplies two numbers.\n\nArgs:\n    a (int): The first number.\n    b (int): The second number.\n\nReturns:\n    int: The product of the two numbers.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x000001243C674720>)

In [16]:
# little more authority on your tool
# Create struct tool directly
from langchain_core.tools import StructuredTool

In [22]:
#validation is going to be a pydantic class
class WeatherInput(BaseModel):
    city: str

In [17]:
def get_weather(city: str) -> str:
    """
    Fetches real-time weather data for a given city.

    Args:
        city (str): The name of the city.
    Returns:
        str: A string describing the current weather in the city.
    """
    return f"The weather in {city} is sunny."

In [ ]:
# this is one of the way of creating a tool
# little more authority , little more validation
weather_tool = StructuredTool.from_function(
    func=get_weather,
    name="get_weather",
    description="Fetches real-time weather data for a city",
    args_schema=WeatherInput # pydantic object as a validation or pydantic class as a validation
)

In [24]:
# One more way of creating this class

In [ ]:
class WeatherInput(BaseModel):
    city: str = Field(..., description="City Name")
    units: str = Field("metric", description="metric or imperial")

# another way to use tool
# imported Structured tool over here now it (GetWeatherTool) will be also considered as a tool
class GetWeatherTool(StructuredTool):
    name: ClassVar[str] = "get_weather"
    description: ClassVar[str] = ("Fetches weather data for a city")
    args_schema: ClassVar[Type[BaseModel]] = WeatherInput

    def _run(self, city: str, units: str = "metric") -> str:
        return get_weather(city,units)